In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import numpy as np
import cv2
from google.colab.patches import cv2_imshow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense
import os
import PIL

In [2]:
path = '/content/drive/MyDrive/Visão Computacional Guia Completo/Datasets/cat_dog_2.zip'
zip_objetict = zipfile.ZipFile(file = path, mode = 'r')
zip_objetict.extractall('./')
zip_objetict.close()

In [3]:
batch_size = 32
img_height = 180
img_width = 180

In [4]:
data_train = '/content/cat_dog_2/training_set'
data_test = '/content/cat_dog_2/test_set'

In [5]:
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_train,
  image_size=(img_height, img_width),
  batch_size=batch_size)

Found 4000 files belonging to 2 classes.


In [6]:
val_ds = tf.keras.utils.image_dataset_from_directory(
  data_test,
  image_size=(img_height, img_width),
  batch_size=batch_size)

Found 1000 files belonging to 2 classes.


In [7]:
class_names = train_ds.class_names
print(class_names)

['cat', 'dog']


In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [9]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [10]:
normalization_layer = layers.Rescaling(1./255)
normalized_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(normalized_ds))
first_image = image_batch[0]
print(np.min(first_image), np.max(first_image))

0.0 1.0


In [24]:
num_classes = len(class_names)

model = Sequential([
  keras.Sequential(
  [
    layers.RandomFlip("horizontal",
                      input_shape=(img_height,
                                  img_width,
                                  3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
  ]
),

   layers.Rescaling(1./255),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.2),
  layers.Flatten(),

  layers.Dense(128, activation='relu'),

  layers.Dense(num_classes)
])

In [25]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [26]:
epochs = 30
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

Epoch 1/30
125/125 [==============================] - 16s 30ms/step - loss: 0.6976 - accuracy: 0.5713 - val_loss: 0.6779 - val_accuracy: 0.5890
Epoch 2/30
125/125 [==============================] - 4s 30ms/step - loss: 0.6406 - accuracy: 0.6425 - val_loss: 0.6481 - val_accuracy: 0.6250
Epoch 3/30
125/125 [==============================] - 3s 28ms/step - loss: 0.6181 - accuracy: 0.6525 - val_loss: 0.6949 - val_accuracy: 0.6240
Epoch 4/30
125/125 [==============================] - 4s 28ms/step - loss: 0.5977 - accuracy: 0.6950 - val_loss: 0.5940 - val_accuracy: 0.6910
Epoch 5/30
125/125 [==============================] - 4s 30ms/step - loss: 0.5664 - accuracy: 0.7122 - val_loss: 0.6749 - val_accuracy: 0.6260
Epoch 6/30
125/125 [==============================] - 4s 29ms/step - loss: 0.5460 - accuracy: 0.7205 - val_loss: 0.6242 - val_accuracy: 0.6510
Epoch 7/30
125/125 [==============================] - 4s 29ms/step - loss: 0.5360 - accuracy: 0.7355 - val_loss: 0.6096 - val_accuracy: 0.704